# Environment setup

In [ ]:
"""
# Create the directory ~/miniconda3 if it does not already exist
mkdir -p ~/miniconda3

# Download the latest Miniconda3 installer for Linux x86_64 architecture
# and save it as ~/miniconda3/miniconda.sh
wget https://repo.anaconda.com/miniconda/Miniconda3-latest-Linux-x86_64.sh -O ~/miniconda3/miniconda.sh

# Run the Miniconda3 installer script in batch mode (-b), update the installation (-u),
# and specify the installation path as ~/miniconda3 (-p ~/miniconda3)
bash ~/miniconda3/miniconda.sh -b -u -p ~/miniconda3

# Remove the Miniconda3 installer script after the installation is complete
rm ~/miniconda3/miniconda.sh

# initialization:
~/miniconda3/bin/conda init bash

# use immediately in the current shell session
source ~/.bashrc

# check conda environments
conda info --env
# Locate the path to the conda executable
which conda
"""

In [ ]:
"""
conda tos accept --override-channels --channel https://repo.anaconda.com/pkgs/main
conda tos accept --override-channels --channel https://repo.anaconda.com/pkgs/r
conda create -n sgkit_train_env python=3.12 -y
conda activate sgkit_train_env
pip install sgkit numpy pandas pyarrow scipy cupy ipykernel anndata -e git+https://github.com/theislab/cellink.git#egg=cellink 
python -m ipykernel install --user --name sgkit_train_env --display-name "sgkit_train_env"
"""

In [ ]:
"""
conda tos accept --override-channels --channel https://repo.anaconda.com/pkgs/main
conda tos accept --override-channels --channel https://repo.anaconda.com/pkgs/r
conda create -n sgkit_train_env5 python=3.12 -y
conda activate sgkit_train_env5
#pip install --upgrade --force-reinstall bio2zarr[vcf] sgkit numpy pandas pyarrow scipy cupy anndata
#pip install ipykernel
#pip install -e git+https://github.com/theislab/cellink.git#egg=cellink
#pip install -e git+https://github.com/scverse/annbatch#egg=annbatch
#pip install "bio2zarr[vcf]==0.1.0" "sgkit==0.8.0" numpy pandas "pyarrow==23.0.0" scipy "cupy==13.6.0" "anndata==0.12.10"  ipykernel -e git+https://github.com/theislab/cellink.git#egg=cellink  
pip install bio2zarr[vcf] sgkit numpy pandas pyarrow scipy cupy ipykernel 
python -m ipykernel install --user --name sgkit_train_env5 --display-name "sgkit_train_env5"
###
pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121  --no-user
pip install anndata annbatch -e git+https://github.com/theislab/cellink.git#egg=cellink  
"""

# Imports

In [ ]:
import os
from tqdm import tqdm
import anndata as ad
import zarr
import h5py
from cellink.io import to_plink, read_sgkit_zarr

# Dataset generation and saving

## Data conversion

In [ ]:
!bash vcf_to_zarr.sh

## Dataset writing

In [ ]:
gdata_list = []
for index in tqdm(range(0, 9)): 
    vcz_file = f"./vcf_files/batch_{index}.vcz"

    gdata = read_sgkit_zarr(vcz_file)  

    gdata.X = gdata.X.astype("uint8")
    gdata_list.append(gdata)

print("Concatenating datasets...")
gdata = ad.concat(gdata_list, axis=1)

gdata = gdata[:, :2_000_000]
gdata

In [ ]:
del gdata.varm["filter"]

In [ ]:
gdata.var = gdata.var[['chrom', 'pos', 'a0', 'a1', 'AN', 'AS_QUALapprox', 'QUALapprox']]
gdata.var

In [5]:
del gdata_list

In [ ]:
gdata.X

In [7]:
import zarr
zarr.config.set({"codec_pipeline.path": "zarrs.ZarrsCodecPipeline", "array.target_shard_size_bytes": 256 * 1024 * 1024})
import anndata
anndata.settings.zarr_write_format = 3
anndata.settings.auto_shard_zarr_v3 = True

In [ ]:
import dask
dask.config.set(num_workers=8)

In [ ]:
gdata.write_h5ad("aou_2M_combined.h5ad", compression="gzip")

In [ ]:
from annbatch import DatasetCollection
import zarr
import os

zarr.config.set(
  {"codec_pipeline.path": "zarrs.ZarrsCodecPipeline"}
)

collection = DatasetCollection(
  zarr.open(os.path.join(f"aou_2M_combined_shuffled.zarr"), mode="w")
)
collection.add_adatas(
  adata_paths=[
      os.path.join(f"aou_2M_combined.h5ad")
  ],
  shuffle=True, # shuffling is needed if you want to use chunked access, but is the default
  n_obs_per_dataset=10000,
)

## Upload data to Google bucket

In [ ]:
my_bucket = os.getenv('WORKSPACE_BUCKET')
print("Workspace bucket path:", my_bucket)

In [ ]:
local_h5ad = "aou_2M_combined.h5ad"

os.system(f"gsutil cp {local_h5ad} {my_bucket}/")

In [ ]:
local_zarr_dir = "aou_2M_combined_shuffled.zarr"

os.system(f"gsutil -m cp -r {local_zarr_dir} {my_bucket}/")

In [ ]:
os.system(f"gsutil ls {my_bucket}/")